# Historical analysis notebook
Code-only copy. Outputs and original narrative cells were removed. See docs/reproduction.md for the revised analysis and execution order.


In [ ]:
from pathlib import Path
import re
import pandas as pd

PROJECT_ROOT = Path.cwd().resolve()
if not (PROJECT_ROOT / "data").exists():
    PROJECT_ROOT = PROJECT_ROOT.parent.resolve()

RAW_DATA_DIR = PROJECT_ROOT / "data/raw_follow_up_subset"
VAR_EXPLANATIONS_PATH = PROJECT_ROOT / "data/_variable_explanations.csv"
PROCESSED_DIR = PROJECT_ROOT / "data/processed_follow_up"
PROCESSED_DIR.mkdir(parents=True, exist_ok=True)

RAW_DATA_DIR, PROCESSED_DIR

In [ ]:
var_explanations = None
for enc in ["utf-8", "latin-1", "cp1252"]:
    try:
        var_explanations = pd.read_csv(VAR_EXPLANATIONS_PATH, sep=";", engine="python", encoding=enc)
        break
    except Exception:
        continue

if var_explanations is None:
    raise RuntimeError("Could not read variable explanations file with supported encodings.")

if {"Variable", "Description"}.issubset(var_explanations.columns):
    column_map = {row["Variable"]: f"{row['Variable']} ({row['Description']})" for _, row in var_explanations.iterrows()}
else:
    column_map = {}

var_explanations[["Variable", "Description"]].head(10)

In [ ]:
# Read all follow-up Excel files into a dictionary keyed by subject id (e.g., CT_01)
data_frames = {}
source_files = {}

for file_path in sorted(RAW_DATA_DIR.glob("*.xlsx")):
    match = re.search(r"(CT_\d+)_", file_path.name)
    if not match:
        print(f"Warning: could not parse subject id from {file_path.name}; skipping.")
        continue

    subject_id = match.group(1)
    if subject_id in data_frames:
        print(f"Warning: duplicate follow-up file for {subject_id}. Replacing {source_files[subject_id]} with {file_path.name}.")

    data_frames[subject_id] = pd.read_excel(file_path)
    source_files[subject_id] = file_path.name

print(f"Loaded {len(data_frames)} follow-up recordings.")
pd.Series(source_files, name="source_file").sort_index()

In [ ]:
target_variables = ["VO2peak_abs", "VO2peak_rel", "PeakPower"]
target_aliases = {
    "VO2peak_abs": ["VO2peak..absolut.", "VO2peak abs", "VO2peak_abs "],
    "VO2peak_rel": ["VO2peak..relativ.", "VO2peak rel", "VO2peak_rel "],
    "PeakPower": ["aerobic.power", "Peak Power", "PeakPower "],
}

def resolve_column_name(df, base_name):
    candidates = []

    descriptive_name = column_map.get(base_name)
    if descriptive_name:
        candidates.append(descriptive_name)

    candidates.append(base_name)
    candidates.extend(target_aliases.get(base_name, []))

    seen = set()
    for candidate in candidates:
        if candidate in seen:
            continue
        seen.add(candidate)
        if candidate in df.columns:
            return candidate

    return None

export_summary = []

for subject_id, df in data_frames.items():
    print(f"Processing {subject_id} ({source_files[subject_id]})...")

    resolved_targets = {}
    for target_name in target_variables:
        resolved_column = resolve_column_name(df, target_name)

        if resolved_column is None:
            print(f"  Warning: {target_name} not found; skipping this target.")
            continue

        non_null_values = df[resolved_column].dropna()
        if non_null_values.empty:
            print(f"  Warning: {target_name} found in {resolved_column}, but only NaN values.")
            continue

        resolved_targets[target_name] = {
            "source_column": resolved_column,
            "value": non_null_values.iloc[0],
        }

    if not resolved_targets:
        print(f"  No target columns found for {subject_id}; skipping export.")
        continue

    target_columns_to_drop = [item["source_column"] for item in resolved_targets.values()]
    data_without_targets = df.drop(columns=target_columns_to_drop, errors="ignore")
    targets_df = pd.DataFrame([{k: v["value"] for k, v in resolved_targets.items()}])

    output_data_path = PROCESSED_DIR / f"{subject_id}_data.csv"
    output_targets_path = PROCESSED_DIR / f"{subject_id}_targets.csv"

    data_without_targets.to_csv(output_data_path, index=False)
    targets_df.to_csv(output_targets_path, index=False)

    export_summary.append({
        "subject_id": subject_id,
        "source_file": source_files[subject_id],
        "output_data": output_data_path.name,
        "output_targets": output_targets_path.name,
        "resolved_targets": ", ".join([f"{k}<-{v['source_column']}" for k, v in resolved_targets.items()]),
    })

print(f"Exported {len(export_summary)} subjects to {PROCESSED_DIR}.")

In [ ]:
summary_df = pd.DataFrame(export_summary).sort_values("subject_id")
summary_df